# Chapter 10 — pandas: tables of data

**Goal:** load a table, look at it, select rows and columns, deal with missing values, compute group statistics
and conditional frequencies, and turn categories into numbers for a model. These are the steps of the Titanic part of the lab.

**Where you'll meet this in MLMA Lab 1:** the *Pandas* section and the whole *Titanic* analysis
(`read_csv`, `info`, `isnull().sum()`, `fillna`, `groupby`, `concat`, boolean filtering, one-hot encoding).

In [ ]:
import io
import numpy as np
import pandas as pd

## 10.1 Series and DataFrame

- A **`Series`** is a 1-D array with an **index** (labels).
- A **`DataFrame`** is a table: a dict of Series sharing one index. Each column has a single dtype.

The data below is a small **made-up** sample with the same columns as the Titanic dataset. It's embedded as text,
so no internet is needed: `io.StringIO` makes a string behave like a file, which `pd.read_csv` can read.
Empty fields become `NaN` (missing).

In [ ]:
CSV = """PassengerId,Survived,Pclass,Sex,Age,Fare,Embarked
1,0,3,male,22,7.25,S
2,1,1,female,38,71.28,C
3,1,3,female,26,7.92,S
4,1,1,female,35,53.10,S
5,0,3,male,35,8.05,S
6,0,3,male,,8.46,Q
7,0,1,male,54,51.86,S
8,0,3,male,2,21.08,S
9,1,3,female,27,11.13,S
10,1,2,female,14,30.07,C
11,1,3,female,4,16.70,S
12,1,1,female,58,26.55,S
13,0,3,male,20,8.05,S
14,0,3,male,39,31.28,S
15,0,3,female,14,7.85,S
16,1,2,female,55,16.00,
17,0,3,male,2,29.13,Q
18,1,2,male,,13.00,S
19,0,3,female,31,18.00,S
20,1,3,female,,7.23,C
21,0,2,male,35,26.00,S
22,1,2,male,34,13.00,S
23,1,3,female,15,8.03,Q
24,1,1,male,28,35.50,S
25,0,3,female,8,21.08,S
26,1,3,female,38,31.39,S
27,0,3,male,,7.23,C
28,0,1,male,19,263.00,S
29,1,3,female,,7.88,Q
30,0,3,male,,7.90,S
"""
raw = pd.read_csv(io.StringIO(CSV))

s = pd.Series([1, 2, 3, 4], index=["a", "b", "c", "d"])
print(s["c"], s.mean())

raw.head()

## 10.2 First look at a DataFrame

| code | shows |
|---|---|
| `df.head(n)`, `df.tail(n)` | first / last rows |
| `df.shape`, `df.columns`, `df.dtypes` | size, column names, types |
| `df.info()` | columns, non-null counts, dtypes |
| `df.describe()` | count, mean, std, quartiles of numeric columns |
| `df["col"].value_counts()` | counts per category (`normalize=True` gives proportions) |
| `df.isnull().sum()` | missing values per column |

In Jupyter, a DataFrame on the **last line** of a cell is displayed as a nice table. Use `display(df)` for others.

In [ ]:
raw.info()
print(raw.shape)
print(raw.isnull().sum())
raw.describe()

## 10.3 Selecting data

| code | gives |
|---|---|
| `df["Age"]` | one column (a Series) |
| `df[["Age", "Fare"]]` | several columns (a DataFrame, note the **double** brackets) |
| `df[df["Age"] > 30]` | rows where the condition holds (a boolean mask, like NumPy) |
| `df[(df["Sex"] == "female") & (df["Pclass"] == 1)]` | combined conditions: `&`, `\|`, `~`, with parentheses |
| `df.loc[row_labels, col_labels]` | select **by label** (slices include the end!) |
| `df.iloc[row_pos, col_pos]` | select **by position** (like NumPy) |
| `df["new"] = ...` | add or overwrite a column |

**pandas 3 note:** never write chained assignments like `df[df.Age > 30]["Fare"] = 0`. They don't change `df`.
Use `df.loc[df["Age"] > 30, "Fare"] = 0` instead.

In [ ]:
df = raw.copy()
print(df["Age"].mean())
print(df[df["Sex"] == "female"]["Survived"].mean())     # the lab's survival-rate pattern
df["FarePerYear"] = df["Fare"] / df["Age"]
df.loc[df["Pclass"] == 1, ["Sex", "Age", "Fare"]].head(3)

## 10.4 Missing data

- `df.isnull()` / `df.isna()` gives a boolean table. `.sum()` counts per column.
- `df["col"].fillna(value)` returns a filled column. **Assign it back**: `df["col"] = df["col"].fillna(...)`.
- `df.dropna()` drops rows with any missing value (`subset=[...]` to restrict to some columns).
- Fill numeric columns with the `median()` (robust to outliers) and categorical ones with the `mode()`.
  `mode()` returns a Series, because there can be ties, so take `.iloc[0]`.

## 10.5 Grouping and aggregation

`df.groupby("col")["target"].mean()` splits the rows by `col`, then computes the mean of `target` in each group.
On a 0/1 column, that mean is a **conditional probability**: $\hat P(\text{Survived}=1 \mid \text{Sex})$.

`pd.crosstab(a, b, normalize="index")` makes a table of conditional frequencies $\hat P(b \mid a)$.

In [ ]:
print(df.groupby("Sex")["Survived"].mean())
print(df.groupby(["Pclass", "Sex"])["Survived"].agg(["mean", "count"]))
pd.crosstab(df["Pclass"], df["Survived"], normalize="index")

## 10.6 Combining tables, encoding, I/O and plotting

- `pd.concat([df1, df2])` stacks rows (`axis=1` puts tables side by side).
- `pd.merge(left, right, on="key", how="left")` joins tables on a key column, like SQL.
- `pd.get_dummies(df, columns=["Sex"], drop_first=True)` does one-hot encoding (the lab uses scikit-learn's `OneHotEncoder` for the same thing).
- `df.to_csv("f.csv", index=False)` / `pd.read_csv("f.csv")`.
- `df.to_numpy()` turns a table into a NumPy array for models.
- `df.plot(kind="bar")`, `df["Age"].hist()`: quick plots that use Matplotlib underneath.

---
# Exercises

All exercises start from `raw`. When an exercise changes data, it works on its own `.copy()`.

### Exercise 10.1: Series
Create `rolls`, a Series with values `[4, 6, 2, 6]` and index `["r1", "r2", "r3", "r4"]`. Then:
- `third`: the value at label `"r3"`
- `n_sixes`: the number of sixes (boolean mask + `.sum()`)
- `as_array`: the values as a NumPy array (`.to_numpy()`)

In [ ]:
rolls = pd.Series([4, 6, 2, 6], index=["r1", "r2", "r3", "r4"])
third = rolls["r3"]
n_sixes = (rolls == 6).sum()
as_array = rolls.to_numpy()

In [ ]:
assert list(rolls.index) == ["r1", "r2", "r3", "r4"]
assert third == 2 and n_sixes == 2
assert isinstance(as_array, np.ndarray) and as_array.tolist() == [4, 6, 2, 6]
print("Exercise 10.1 passed")

### Exercise 10.2: Build a DataFrame from a dict
Create `models` with columns `"name"` (`["tree", "logreg", "knn"]`), `"accuracy"` (`[0.78, 0.81, 0.74]`)
and `"train_time"` (`[1.2, 0.4, 0.1]`). Then:
- `n_rows, n_cols`: from `.shape`
- `col_names`: the column names as a list

In [ ]:
models = pd.DataFrame({"name": ["tree", "logreg", "knn"],
                       "accuracy": [0.78, 0.81, 0.74],
                       "train_time": [1.2, 0.4, 0.1]})
n_rows, n_cols = models.shape
col_names = list(models.columns)
models

In [ ]:
assert (n_rows, n_cols) == (3, 3)
assert col_names == ["name", "accuracy", "train_time"]
assert models.loc[1, "name"] == "logreg"
print("Exercise 10.2 passed")

### Exercise 10.3: Exploring the passenger data
Using `raw`:
- `n_passengers`: the number of rows
- `missing`: a Series with the number of missing values per column
- `cols_with_missing`: a list of the names of columns that have at least one missing value
- `class_counts`: how many passengers are in each `Pclass` (`value_counts()`)

In [ ]:
n_passengers = len(raw)
missing = raw.isnull().sum()
cols_with_missing = list(missing[missing > 0].index)
class_counts = raw["Pclass"].value_counts()

In [ ]:
assert n_passengers == 30
assert missing["Age"] == 6 and missing["Embarked"] == 1 and missing["Fare"] == 0
assert sorted(cols_with_missing) == ["Age", "Embarked"]
assert class_counts[3] == 19 and class_counts.sum() == 30
print("Exercise 10.3 passed")

### Exercise 10.4: Selecting columns and adding new ones
On a copy `df = raw.copy()`:
- `ages`: the `Age` column (a Series)
- `small`: a DataFrame with only `Sex`, `Age` and `Survived`
- add a column `"IsChild"`, True when `Age < 16` (missing ages compare as False)
- add a column `"LogFare"` = `np.log(Fare)`

In [ ]:
df = raw.copy()
ages = df["Age"]
small = df[["Sex", "Age", "Survived"]]
df["IsChild"] = df["Age"] < 16
df["LogFare"] = np.log(df["Fare"])

In [ ]:
assert isinstance(ages, pd.Series) and isinstance(small, pd.DataFrame)
assert list(small.columns) == ["Sex", "Age", "Survived"]
assert df["IsChild"].sum() == 7
assert np.allclose(np.exp(df["LogFare"]), df["Fare"])
print("Exercise 10.4 passed")

### Exercise 10.5: Filtering rows
Using boolean masks on `raw`:
- `women`: all rows with `Sex == "female"`
- `first_class_women`: women in `Pclass` 1
- `young_or_rich`: passengers younger than 10 **or** with a fare above 50
- `p_surv_women`: the survival rate among women (mean of `Survived` within `women`)

In [ ]:
women = raw[raw["Sex"] == "female"]
first_class_women = raw[(raw["Sex"] == "female") & (raw["Pclass"] == 1)]
young_or_rich = raw[(raw["Age"] < 10) | (raw["Fare"] > 50)]
p_surv_women = women["Survived"].mean()

In [ ]:
assert len(women) == 15 and len(first_class_women) == 3
assert sorted(young_or_rich["PassengerId"]) == [2, 4, 7, 8, 11, 17, 25, 28]
assert np.isclose(p_surv_women, 12 / 15)
print("Exercise 10.5 passed")

### Exercise 10.6: `loc` and `iloc`
- `first_three`: the first 3 rows and the first 2 columns, **by position** (`iloc`)
- `fare_of_10`: the `Fare` of the passenger whose `PassengerId` is 10. First make a copy indexed by id:
  `by_id = raw.set_index("PassengerId")`, then use `.loc`.
- `ages_5_to_7`: the `Age` values for passenger ids 5 through 7 with `.loc` (label slices **include** the end)

In [ ]:
first_three = raw.iloc[:3, :2]
by_id = raw.set_index("PassengerId")
fare_of_10 = by_id.loc[10, "Fare"]
ages_5_to_7 = by_id.loc[5:7, "Age"]

In [ ]:
assert first_three.shape == (3, 2)
assert fare_of_10 == 30.07
assert len(ages_5_to_7) == 3 and ages_5_to_7.iloc[0] == 35 and np.isnan(ages_5_to_7.iloc[1])
print("Exercise 10.6 passed")

### Exercise 10.7: Filling missing values (exactly as in the lab)
On `df = raw.copy()`:
- fill missing `Age` with the **median** age
- fill missing `Embarked` with the **mode** (`.mode().iloc[0]`)
- set `remaining_missing` to the total number of missing values left in `df` (should be 0)

Then think about it: why the median for `Age`, and why not the mean for `Embarked`?

In [ ]:
df = raw.copy()
df["Age"] = df["Age"].fillna(df["Age"].median())
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode().iloc[0])
remaining_missing = df.isnull().sum().sum()

In [ ]:
assert remaining_missing == 0
assert (df.loc[raw["Age"].isnull(), "Age"] == raw["Age"].median()).all()
assert df.loc[15, "Embarked"] == "S"
print("Exercise 10.7 passed")

### Exercise 10.8: Group statistics
Using `raw`:
- `surv_by_class`: survival rate per `Pclass` (a Series indexed by class)
- `mean_fare_by_port`: mean `Fare` per `Embarked`
- `stats_by_sex`: a DataFrame with the **mean and count** of `Survived` per `Sex` (use `.agg(["mean", "count"])`)

**Predict first:** which class has the highest survival rate?

In [ ]:
surv_by_class = raw.groupby("Pclass")["Survived"].mean()
mean_fare_by_port = raw.groupby("Embarked")["Fare"].mean()
stats_by_sex = raw.groupby("Sex")["Survived"].agg(["mean", "count"])
stats_by_sex

In [ ]:
assert np.isclose(surv_by_class[1], 4 / 6) and np.isclose(surv_by_class[3], 7 / 19)
assert set(mean_fare_by_port.index) == {"C", "Q", "S"}
assert list(stats_by_sex.columns) == ["mean", "count"] and stats_by_sex.loc["male", "count"] == 15
print("Exercise 10.8 passed")

### Exercise 10.9: Empirical probabilities
Treat the table as a sample and estimate:
- `pmf_class`: $\hat P(\text{Pclass} = c)$ for each class (`value_counts(normalize=True)`, sorted by index)
- `cond`: the table $\hat P(\text{Survived} \mid \text{Sex})$ with `pd.crosstab(raw["Sex"], raw["Survived"], normalize="index")`
- `p_surv_given_male`: read $\hat P(\text{Survived} = 1 \mid \text{male})$ from `cond` with `.loc`
- `p_male_given_surv`: $\hat P(\text{male} \mid \text{Survived} = 1)$. That's a *different* conditional probability.
  Compute it by filtering survivors.

In [ ]:
pmf_class = raw["Pclass"].value_counts(normalize=True).sort_index()
cond = pd.crosstab(raw["Sex"], raw["Survived"], normalize="index")
p_surv_given_male = cond.loc["male", 1]
p_male_given_surv = (raw.loc[raw["Survived"] == 1, "Sex"] == "male").mean()
cond

In [ ]:
assert np.isclose(pmf_class.sum(), 1) and list(pmf_class.index) == [1, 2, 3]
assert np.allclose(cond.sum(axis=1), 1)
assert np.isclose(p_surv_given_male, 3 / 15)
assert np.isclose(p_male_given_surv, 3 / 15)
print("Exercise 10.9 passed. Notice that P(A|B) and P(B|A) are not the same thing")

### Exercise 10.10: Sorting, duplicates, dropping columns
- `top_fares`: the 3 rows with the highest `Fare`, highest first (`sort_values(..., ascending=False)` then `.head(3)`)
- `dedup`: `dupes` without duplicate rows (`drop_duplicates()`)
- `slim`: `raw` without the columns `PassengerId` and `Fare` (`drop(columns=[...])`)

In [ ]:
dupes = pd.DataFrame({"A": [1, 2, 2], "B": [4, 5, 5], "C": ["p", "q", "q"]})
top_fares = raw.sort_values("Fare", ascending=False).head(3)
dedup = dupes.drop_duplicates()
slim = raw.drop(columns=["PassengerId", "Fare"])

In [ ]:
assert list(top_fares["PassengerId"]) == [28, 2, 4]
assert len(dedup) == 2
assert "Fare" not in slim.columns and "PassengerId" not in slim.columns and slim.shape == (30, 5)
print("Exercise 10.10 passed")

### Exercise 10.11: concat and merge
- `stacked`: `part1` and `part2` stacked **on top of each other** with `pd.concat`, using `ignore_index=True`
- `with_port`: `raw` merged with `ports` on `Embarked` (`how="left"`), so every passenger gets a `PortName` column
  (passengers with missing `Embarked` get `NaN`)

In [ ]:
part1 = raw.iloc[:10]
part2 = raw.iloc[10:]
ports = pd.DataFrame({"Embarked": ["S", "C", "Q"],
                      "PortName": ["Southampton", "Cherbourg", "Queenstown"]})
stacked = pd.concat([part1, part2], ignore_index=True)
with_port = pd.merge(raw, ports, on="Embarked", how="left")

In [ ]:
assert stacked.shape == raw.shape and list(stacked.index) == list(range(30))
assert len(with_port) == 30 and "PortName" in with_port.columns
assert with_port.loc[with_port["PassengerId"] == 2, "PortName"].item() == "Cherbourg"
assert with_port["PortName"].isnull().sum() == 1
print("Exercise 10.11 passed")

### Exercise 10.12: Categories to numbers
Models need numbers. On `df = raw.copy()` (fill `Embarked` with `"S"` first):
- add `"IsMale"`: 1 for male, 0 for female, using `.map({"male": 1, "female": 0})`
- `encoded`: `pd.get_dummies(df, columns=["Embarked"], drop_first=True, dtype=int)`
- `X`: a NumPy array (`.to_numpy()`) with the columns `Pclass`, `IsMale`, `Embarked_Q`, `Embarked_S` from `encoded`

Why does `drop_first=True` lose no information?

In [ ]:
df = raw.copy()
df["Embarked"] = df["Embarked"].fillna("S")
df["IsMale"] = df["Sex"].map({"male": 1, "female": 0})
encoded = pd.get_dummies(df, columns=["Embarked"], drop_first=True, dtype=int)
X = encoded[["Pclass", "IsMale", "Embarked_Q", "Embarked_S"]].to_numpy()

In [ ]:
assert df["IsMale"].sum() == 15
assert "Embarked_C" not in encoded.columns and {"Embarked_Q", "Embarked_S"} <= set(encoded.columns)
assert isinstance(X, np.ndarray) and X.shape == (30, 4)
assert X[1].tolist() == [1, 0, 0, 0], "passenger 2: first class, female, embarked at C"
print("Exercise 10.12 passed")

### Exercise 10.13: Binning and a quick plot
- add a column `"AgeGroup"` with `pd.cut(df["Age"], bins=[0, 12, 18, 60, 100], labels=["child", "teen", "adult", "senior"])`
- `surv_by_age`: survival rate per `AgeGroup` (pass `observed=True` to `groupby`)
- plot it as a bar chart: `ax = surv_by_age.plot(kind="bar")`, with the y-label `"survival rate"`

In [ ]:
import matplotlib.pyplot as plt
df = raw.copy()
df["AgeGroup"] = pd.cut(df["Age"], bins=[0, 12, 18, 60, 100], labels=["child", "teen", "adult", "senior"])
surv_by_age = df.groupby("AgeGroup", observed=True)["Survived"].mean()
ax = surv_by_age.plot(kind="bar")
ax.set_ylabel("survival rate")
plt.show()

In [ ]:
assert list(surv_by_age.index) == ["child", "teen", "adult"], "no seniors in this sample"
assert np.isclose(surv_by_age["teen"], 2 / 3)
assert len(ax.patches) == 3 and ax.get_ylabel() == "survival rate"
print("Exercise 10.13 passed")

---
**Chapter 10 done.** Last stop: Chapter 11, the capstone, where everything comes together in an MLMA-style pipeline.